# Can a linear model on the LLM's token probabilities predict the BERT sentiment score?

Feasibility test for the differentiable loop. The plan is to replace the
text → RoBERTa sensor with a **linear read-out of Qwen's next-token probabilities**,
which never passes through text and so is differentiable end to end.

This notebook asks one question: **how much of RoBERTa's score can such a read-out
recover?**

**Data.** Qwen2.5-0.5B-Instruct answers the PoC prompt at each of the 7 instruction
rungs, which gives a spread of sentiment. For each response we record:

- the full next-token distribution at every generated position, stored as a sparse
  top-k and **averaged over positions**;
- the sampled tokens (for a baseline);
- RoBERTa's score of the decoded text, as both `prob` (P(pos) − P(neg)) and `logit`.

**The model.** Ridge regression on the position-averaged probability vector. It is
equivalent to giving each vocabulary token a learned sentiment weight and taking the
expected weight under the model's own distribution, averaged over the response.

**Sampling is untruncated** (`top_p = 1`, `top_k = 0`, `repetition_penalty = 1`), so
the recorded probabilities are exactly the distribution the tokens were drawn from.
Qwen's `generation_config.json` otherwise silently applies `top_k = 20` and
`repetition_penalty = 1.1`.

**Baselines and controls:**

| model | what it knows |
| --- | --- |
| constant | nothing: predicts the training mean |
| level oracle | which instruction was given: predicts that rung's training mean |
| bag of sampled tokens | ridge on the tokens actually emitted, not their probabilities |
| **expected probabilities** | **the proposal** |

The level oracle is the control that matters. The probabilities implicitly reveal the
instruction, so a surrogate could score well just by recognising the rung. The loop
needs more than that: it needs the surrogate to track the **sample-to-sample
variation within a rung**, because that variation is the noise that drives the chain.
The metrics *skill over oracle* and *within-rung r* measure exactly this.

In [ ]:
import os, sys, time, hashlib
from pathlib import Path

os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt

# repo root = the directory containing the `humancompatible` package
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "humancompatible").is_dir())
sys.path.insert(0, str(ROOT))

from humancompatible.interconnect.simulators.logic.agents.llm_agent import (
    NUM_LEVELS, NEUTRALITY_LADDER, BASE_PROMPT, render_prompt, SentimentScorer,
)

torch.set_num_threads(os.cpu_count())
print(ROOT)

## Configuration

In [ ]:
N_PER_LEVEL    = int(os.environ.get("N_PER_LEVEL", 100))     # 7 x 200 = 1400 responses, ~15-20 min on CPU
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", 32))  # as in the PoC
BATCH          = 25
TOP_K          = 256          # tokens kept per position; retained mass is reported below
TEMPERATURE    = 1.0
SEED           = 0
DEVICE         = "cpu"        # the iGPU OOMs, see README section 14
MODEL_NAME     = "Qwen/Qwen2.5-0.5B-Instruct"
TEST_FRAC      = 0.2

tag = hashlib.md5(f"{MODEL_NAME}|{BASE_PROMPT}|{NEUTRALITY_LADDER}|{N_PER_LEVEL}|"
                  f"{MAX_NEW_TOKENS}|{TOP_K}|{TEMPERATURE}|{SEED}".encode()).hexdigest()[:10]
CACHE = ROOT / "examples" / "llm_poc" / "surrogate_data" / tag
print("cache:", CACHE)

## 1. Generate responses and record the token distributions

Every response at a rung uses the same prompt, so a batch needs no padding. The
generation logits are raw and the sampling is untruncated, so the softmax of the
logits is exactly the distribution each token was drawn from.

A response ends at its first EOS. The EOS position is included, because the
probability of stopping is part of the distribution. Positions after EOS are
discarded.

In [ ]:
def load_llm():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(DEVICE).eval()
    return tok, llm


@torch.inference_mode()
def generate_batch(tok, llm, level, n):
    chat = [{"role": "user", "content": render_prompt(level)}]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    enc = tok([text] * n, return_tensors="pt").to(DEVICE)
    out = llm.generate(
        **enc, do_sample=True, temperature=TEMPERATURE, top_p=1.0, top_k=0,
        repetition_penalty=1.0, max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tok.pad_token_id, output_logits=True, return_dict_in_generate=True,
    )
    gen = out.sequences[:, enc["input_ids"].shape[1]:]                  # (n, T)
    probs = torch.softmax(torch.stack(out.logits).float() / TEMPERATURE, dim=-1)  # (T, n, V)
    topv, topi = probs.topk(TOP_K, dim=-1)                             # (T, n, K)

    eos = torch.tensor(llm.generation_config.eos_token_id).reshape(-1)
    is_eos = torch.isin(gen, eos)
    T = gen.shape[1]
    lengths = torch.where(is_eos.any(1), is_eos.float().argmax(1) + 1, torch.full((n,), T))

    rows = []
    for b in range(n):
        L = int(lengths[b])
        rows.append(dict(
            level=level,
            length=L,
            text=tok.decode(gen[b, :L], skip_special_tokens=True).strip(),
            exp_cols=topi[:L, b].reshape(-1).numpy(),
            exp_vals=(topv[:L, b].reshape(-1) / L).numpy(),
            kept_mass=float(topv[:L, b].sum(-1).mean()),
            bag_cols=gen[b, :L].numpy(),
        ))
    return rows, probs.shape[-1]


def build_dataset():
    torch.manual_seed(SEED)
    tok, llm = load_llm()
    rows, t0 = [], time.time()
    for level in range(NUM_LEVELS):
        for start in range(0, N_PER_LEVEL, BATCH):
            batch, V = generate_batch(tok, llm, level, min(BATCH, N_PER_LEVEL - start))
            rows += batch
        print(f"level {level}: {len(rows)} responses, {time.time() - t0:.0f} s")
    del llm

    def to_csr(cols_key, vals_key=None):
        r = np.concatenate([np.full(len(x[cols_key]), i) for i, x in enumerate(rows)])
        c = np.concatenate([x[cols_key] for x in rows])
        v = (np.concatenate([x[vals_key] for x in rows]) if vals_key else
             np.concatenate([np.full(x["length"], 1.0 / x["length"]) for x in rows]))
        return sp.coo_matrix((v, (r, c)), shape=(len(rows), V)).tocsr()   # duplicates summed

    X_exp, X_bag = to_csr("exp_cols", "exp_vals"), to_csr("bag_cols")
    meta = pd.DataFrame({k: [x[k] for x in rows] for k in ("level", "length", "text", "kept_mass")})

    scorer = SentimentScorer(device=DEVICE)
    meta["y_prob"] = scorer.score(meta.text.tolist(), readout="prob")
    meta["y_logit"] = scorer.score(meta.text.tolist(), readout="logit")

    CACHE.mkdir(parents=True, exist_ok=True)
    sp.save_npz(CACHE / "X_exp.npz", X_exp)
    sp.save_npz(CACHE / "X_bag.npz", X_bag)
    meta.to_parquet(CACHE / "meta.parquet") if _has_parquet() else meta.to_pickle(CACHE / "meta.pkl")
    return X_exp, X_bag, meta


def _has_parquet():
    try:
        import pyarrow  # noqa: F401
        return True
    except ImportError:
        return False


def load_dataset():
    meta = (pd.read_parquet(CACHE / "meta.parquet") if (CACHE / "meta.parquet").exists()
            else pd.read_pickle(CACHE / "meta.pkl"))
    return sp.load_npz(CACHE / "X_exp.npz"), sp.load_npz(CACHE / "X_bag.npz"), meta


X_exp, X_bag, meta = load_dataset() if (CACHE / "X_exp.npz").exists() else build_dataset()
tok = __import__("transformers").AutoTokenizer.from_pretrained(MODEL_NAME)
print(X_exp.shape, f"nnz/row = {X_exp.nnz / X_exp.shape[0]:.0f}")

## 2. Look at the data

- **Retained mass** is how much probability the top-k keeps. It should be close to 1,
  otherwise the stored features are a biased truncation.
- **The score per rung** shows the target. RoBERTa saturates, so `prob` piles up
  near ±1 within a rung. The `logit` readout does not saturate.

In [ ]:
print(f"retained top-{TOP_K} mass per position:  mean {meta.kept_mass.mean():.4f}, "
      f"min {meta.kept_mass.min():.4f}")
print(f"response length (tokens incl. EOS):  mean {meta.length.mean():.1f}, "
      f"hit max {np.mean(meta.length == MAX_NEW_TOKENS):.0%}")
display(meta.groupby("level")[["y_prob", "y_logit", "length"]].agg(["mean", "std"]).round(3))
display(meta.sample(6, random_state=0)[["level", "y_prob", "text"]])

In [ ]:
# Rungs are ordered positive -> critical: diverging blue <-> red with a gray midpoint.
LEVEL_COLORS = ["#104281", "#2a78d6", "#86b6ef", "#8a8984", "#ef9a9a", "#d9534f", "#9e2a2a"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6})

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, name in zip(axes, ["y_prob", "y_logit"], ["P(pos) − P(neg)", "logit(pos) − logit(neg)"]):
    for k in range(NUM_LEVELS):
        y = meta.y_prob[meta.level == k] if col == "y_prob" else meta.y_logit[meta.level == k]
        ax.scatter(k + rng.uniform(-0.25, 0.25, len(y)), y, s=8, alpha=0.5,
                   color=LEVEL_COLORS[k], linewidths=0)
        ax.plot([k - 0.3, k + 0.3], [y.mean()] * 2, color="#222", lw=2)
    ax.set_xticks(range(NUM_LEVELS))
    ax.set_xlabel("instruction rung (0 = most positive, 6 = most critical)")
    ax.set_title(f"RoBERTa score by rung: {name}", fontsize=10, loc="left")
plt.tight_layout(); plt.show()

## 3. Train/test split and ridge regression

The split is stratified by rung. With several hundred thousand candidate features and
about a thousand training rows, the problem is heavily over-parameterised, so ridge
is fitted in its **dual (kernel) form**: one n × n solve, with the feature centring
folded into the kernel. λ is chosen by 5-fold cross-validation on the training set only.

In [ ]:
def ridge_fit(X, y, lam):
    # Centred ridge in dual form. Returns primal weights over the vocabulary, and the intercept.
    mu = np.asarray(X.mean(0)).ravel()
    a = X @ mu
    Kc = (X @ X.T).toarray() - a[:, None] - a[None, :] + mu @ mu
    ybar = y.mean()
    alpha = np.linalg.solve(Kc + lam * np.eye(len(y)), y - ybar)
    w = X.T @ alpha - mu * alpha.sum()
    return w, ybar - mu @ w


def ridge_predict(model, X):
    w, b = model
    return X @ w + b


def cv_lambda(X, y, strata, lams, folds=5, seed=0):
    rng = np.random.default_rng(seed)
    fold = np.empty(len(y), int)
    for s in np.unique(strata):                       # stratified fold assignment
        idx = rng.permutation(np.flatnonzero(strata == s))
        fold[idx] = np.arange(len(idx)) % folds
    mse = np.zeros(len(lams))
    for f in range(folds):
        tr, va = fold != f, fold == f
        for j, lam in enumerate(lams):
            mse[j] += np.sum((ridge_predict(ridge_fit(X[tr], y[tr], lam), X[va]) - y[va]) ** 2)
    return lams[np.argmin(mse)], mse / len(y)


rng = np.random.default_rng(SEED)
lev = meta.level.to_numpy()
test = np.zeros(len(meta), bool)
for k in range(NUM_LEVELS):
    idx = rng.permutation(np.flatnonzero(lev == k))
    test[idx[: int(round(TEST_FRAC * len(idx)))]] = True
train = ~test
print(f"train {train.sum()}, test {test.sum()}")

LAMS = np.logspace(-9, 1, 21)

In [ ]:
def metrics(y, yhat, lev, oracle_hat):
    sse = np.sum((y - yhat) ** 2)
    d = pd.DataFrame({"y": y, "h": yhat, "k": lev})
    dy = d.y - d.groupby("k").y.transform("mean")      # within-rung deviations
    dh = d.h - d.groupby("k").h.transform("mean")
    return {
        "R2": 1 - sse / np.sum((y - y.mean()) ** 2),
        "r": np.corrcoef(y, yhat)[0, 1],
        "MAE": np.mean(np.abs(y - yhat)),
        "skill over oracle": 1 - sse / np.sum((y - oracle_hat) ** 2),
        "within-rung r": np.corrcoef(dy, dh)[0, 1] if dh.std() > 0 else np.nan,
    }


results, fitted = {}, {}
for target in ["y_prob", "y_logit"]:
    y = meta[target].to_numpy()
    level_mean = pd.Series(y[train]).groupby(lev[train]).mean()
    oracle = level_mean.reindex(lev).to_numpy()
    preds = {"constant": np.full(len(y), y[train].mean()), "level oracle": oracle}
    for name, X in [("bag of sampled tokens", X_bag), ("expected probabilities", X_exp)]:
        lam, _ = cv_lambda(X[train], y[train], lev[train], LAMS)
        if lam in (LAMS[0], LAMS[-1]):
            print(f"warning: {target}/{name}: CV picked lambda at the grid edge ({lam:.0e}); widen LAMS")
        model = ridge_fit(X[train], y[train], lam)
        preds[name] = ridge_predict(model, X)
        fitted[(target, name)] = (model, lam)
    for name, p in preds.items():
        results[(target, name)] = {**metrics(y[test], p[test], lev[test], oracle[test]),
                                   "lambda": fitted.get((target, name), (None, np.nan))[1]}
    fitted[(target, "preds")] = preds

res = pd.DataFrame(results).T
res.index.names = ["target", "model"]
display(res.round(3))

**How to read the table**

- **R²** and **r** measure overall fit on held-out responses.
- **skill over oracle** is the fraction of the level oracle's squared error that the
  model removes. Above 0 means it beats *knowing the instruction*. That is the bar
  that matters.
- **within-rung r** is the correlation between prediction and target after
  subtracting each rung's mean from both. It asks whether the surrogate tracks the
  **noise**, not the rung. Anything near 0 means the surrogate has only learned to
  recognise the prompt.
- If *expected probabilities* is close to *bag of sampled tokens*, taking the
  expectation costs nothing relative to reading the realised tokens. If it is clearly
  worse, the realised path carries information that the averaged distribution loses.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, target in zip(axes, ["y_prob", "y_logit"]):
    y = meta[target].to_numpy()
    p = fitted[(target, "preds")]["expected probabilities"]
    for k in range(NUM_LEVELS):
        m = test & (lev == k)
        ax.scatter(y[m], p[m], s=14, alpha=0.75, color=LEVEL_COLORS[k], linewidths=0,
                   label=f"rung {k}")
    lo, hi = np.min(y[test]), np.max(y[test])
    ax.plot([lo, hi], [lo, hi], color="#888", lw=1, ls="--")
    r = res.loc[(target, "expected probabilities")]
    ax.set_title(f"{target}: held-out, R² = {r.R2:.2f}, within-rung r = {r['within-rung r']:.2f}",
                 fontsize=10, loc="left")
    ax.set_xlabel("RoBERTa score"); ax.set_ylabel("linear surrogate")
axes[1].legend(fontsize=8, frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

## 3b. Small MLP

Ridge does no better than the level oracle, so try a nonlinear read-out. A raw
152k-dimensional input would let an MLP memorise a few hundred rows, so it gets one of
two compact inputs:

- **expected embedding**: the position-averaged probabilities multiplied by Qwen's token
  embedding matrix (896-d). This is still linear in the probabilities, so the sensor stays
  differentiable end to end.
- **top vocab**: the 4096 vocabulary columns carrying the most training mass.

The network is one hidden layer (64, GELU, dropout), trained with AdamW, with early
stopping on a held-out sixth of the training set. Predictions are averaged over 5 seeds.

In [ ]:
import copy
import torch.nn as nn
from transformers import AutoModelForCausalLM

E = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).get_input_embeddings().weight.detach().numpy()
X_emb = np.asarray(X_exp @ E[: X_exp.shape[1]])                 # (n, 896) expected embedding
top_cols = np.argsort(-np.asarray(X_exp[train].sum(0)).ravel())[:4096]
X_top = X_exp[:, top_cols].toarray()
del E


def train_mlp(X, y, hidden=64, dropout=0.2, wd=1e-2, lr=1e-3, epochs=3000, patience=200, seed=0):
    torch.manual_seed(seed)
    idx = np.random.default_rng(seed).permutation(np.flatnonzero(train))
    va, fit = idx[: len(idx) // 6], idx[len(idx) // 6:]
    mu, sd = X[fit].mean(0), X[fit].std(0)
    sd = np.maximum(sd, np.median(sd))                     # floor: rare columns would otherwise explode
    ym, ys = y[fit].mean(), y[fit].std()
    Xt = torch.tensor((X - mu) / sd, dtype=torch.float32)
    yt = torch.tensor((y - ym) / ys, dtype=torch.float32)

    net = nn.Sequential(nn.Linear(X.shape[1], hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    best, state, wait = np.inf, None, 0
    for _ in range(epochs):
        net.train(); opt.zero_grad()
        ((net(Xt[fit]).squeeze(1) - yt[fit]) ** 2).mean().backward(); opt.step()
        net.eval()
        with torch.no_grad():
            v = ((net(Xt[va]).squeeze(1) - yt[va]) ** 2).mean().item()
        if v < best:
            best, state, wait = v, copy.deepcopy(net.state_dict()), 0
        elif (wait := wait + 1) > patience:
            break
    net.load_state_dict(state); net.eval()
    with torch.no_grad():
        return net(Xt).squeeze(1).numpy() * ys + ym


for target in ["y_prob", "y_logit"]:
    y = meta[target].to_numpy()
    oracle = fitted[(target, "preds")]["level oracle"]
    for name, X in [("MLP: expected embedding", X_emb), ("MLP: top vocab", X_top)]:
        p = np.mean([train_mlp(X, y, seed=s) for s in range(5)], axis=0)
        fitted[(target, "preds")][name] = p
        results[(target, name)] = {**metrics(y[test], p[test], lev[test], oracle[test]), "lambda": np.nan}

res = pd.DataFrame(results).T.sort_index(level=0, sort_remaining=False)
res.index.names = ["target", "model"]
display(res.drop(columns="lambda").round(3))

In [ ]:
best = res.loc["y_logit"].drop(["constant", "level oracle"])["skill over oracle"].idxmax()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, target in zip(axes, ["y_prob", "y_logit"]):
    y = meta[target].to_numpy()
    p = fitted[(target, "preds")][best]
    for k in range(NUM_LEVELS):
        m = test & (lev == k)
        ax.scatter(y[m], p[m], s=14, alpha=0.75, color=LEVEL_COLORS[k], linewidths=0, label=f"rung {k}")
    lo, hi = np.min(y[test]), np.max(y[test])
    ax.plot([lo, hi], [lo, hi], color="#888", lw=1, ls="--")
    r = res.loc[(target, best)]
    ax.set_title(f"{best}, {target}: R² = {r.R2:.2f}, within-rung r = {r['within-rung r']:.2f}",
                 fontsize=10, loc="left")
    ax.set_xlabel("RoBERTa score"); ax.set_ylabel("surrogate")
axes[1].legend(fontsize=8, frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

**If the MLP also stalls near the ridge's within-rung r**, the limit is the features,
not the model. Averaging over positions throws away word order, so negation and
contrast ("despite …, it is reliable") are invisible to *any* read-out of this vector.
The next step would then be a per-position read-out: score each position's
distribution, then pool.

## 4. Does more data help?

A learning curve for the proposal at the cross-validated λ. If held-out R² is still
climbing at the full training set, the limit is the data, not the linear model.

In [ ]:
fracs = [0.1, 0.2, 0.35, 0.5, 0.75, 1.0]
tr_idx = np.flatnonzero(train)
curve = {}
for target in ["y_prob", "y_logit"]:
    y = meta[target].to_numpy()
    lam = fitted[(target, "expected probabilities")][1]
    for f in fracs:
        scores = []
        for rep in range(3):
            sub = np.random.default_rng(rep).choice(tr_idx, int(f * len(tr_idx)), replace=False)
            p = ridge_predict(ridge_fit(X_exp[sub], y[sub], lam), X_exp[test])
            scores.append(1 - np.sum((y[test] - p) ** 2) / np.sum((y[test] - y[test].mean()) ** 2))
        curve[(target, f)] = np.mean(scores)

fig, ax = plt.subplots(figsize=(6, 3.4))
for target, color in [("y_prob", "#2a78d6"), ("y_logit", "#eb6834")]:
    xs = [int(f * len(tr_idx)) for f in fracs]
    ys = [curve[(target, f)] for f in fracs]
    ax.plot(xs, ys, color=color, lw=2, marker="o", ms=5)
    ax.annotate(target, (xs[-1], ys[-1]), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=9)
ax.set_xlabel("training responses"); ax.set_ylabel("held-out R²")
ax.set_title("Learning curve: expected-probability surrogate", fontsize=10, loc="left")
plt.tight_layout(); plt.show()

## 5. What did it learn?

These are the learned per-token weights, restricted to tokens that carry probability
in at least 2% of training responses, because weights on rare tokens are mostly
noise. If the surrogate reads sentiment, the lists should be recognisable sentiment
vocabulary. If they are dominated by rung-specific phrasing (echoes of the
instruction, formatting, length), it is recognising the prompt instead.

In [ ]:
target = "y_logit"
(w, b), lam = fitted[(target, "expected probabilities")]
doc_freq = np.asarray((X_exp[train] > 0).sum(0)).ravel() / train.sum()
eligible = np.flatnonzero(doc_freq >= 0.02)
order = eligible[np.argsort(w[eligible])]

def show(ids):
    return pd.DataFrame({"token": [repr(tok.decode([i])) for i in ids],
                         "weight": w[ids].round(2), "doc freq": doc_freq[ids].round(3)})

print(f"{target}, lambda = {lam:.1e}, {len(eligible)} eligible tokens")
display(pd.concat({"most negative": show(order[:20]).reset_index(drop=True),
                   "most positive": show(order[::-1][:20]).reset_index(drop=True)}, axis=1))

## 6. Verdict

Fill in after running. The questions to answer:

1. Does *expected probabilities* beat the **level oracle** (skill > 0) and show a
   clearly non-zero **within-rung r**? If not, a linear read-out of the distribution
   cannot serve as the loop's sensor, and the next step is a small nonlinear head, or a
   probe on hidden states.
2. Which **target** does it fit better? `logit` is expected to fit better because it
   does not saturate. The loop can use either, as long as it uses the same one throughout.
3. Is the learning curve **flat** (the model is the limit) or **rising** (the data is
   the limit)?
4. Do the top tokens look like **sentiment** or like **prompt recognition**?

One caveat on scope. This data spans the space through the *instruction* ladder, but
the loop will actuate through *steering*. Before the surrogate is trusted in the loop,
it must be refitted, or at least re-validated, on steered generations.